In [1]:
# %pip install --upgrade pip

In [2]:
# pip install --force-reinstall git+https://github.com/khanh-at-gex/gex-msgraph.git

In [3]:
# %pip install python-calamine

In [4]:
# Import necessary libraries
from dotenv import load_dotenv

load_dotenv()
from gex_msgraph import GraphClient, FileItem

In [5]:
ROOT_DIR = r"2026\Project Phoenix\04. Working files"

In [6]:
client = GraphClient("das_u1")
# await client.__aenter__()

# items = await client.list_files(ROOT_DIR)
# print(f"Connected. Found {len(items)} items at the drive root.")

In [7]:
import pandas as pd

# 1. All folders (relative to ROOT_DIR)
items = await client.list_files(ROOT_DIR)
df_folders = pd.DataFrame(items)[["name", "path"]].rename(
    columns={"name": "folder_name", "path": "folder_path"}
)
# df_folders

# 2. All .xlsx files (relative to ROOT_DIR)
xlsx_files = await client.walk(ROOT_DIR, pattern="*.xls*")
df_xlsx = pd.DataFrame(xlsx_files)[["name", "path", "modified"]].rename(
    columns={"name": "file_name", "path": "file_path", "modified": "file_modified"}
)
df_xlsx["folder_path"] = (
    df_xlsx["file_path"].str.rsplit("/", n=1).str[0].str.removeprefix(ROOT_DIR + "/")
)

# df_list_files = df_folders.merge(df_xlsx, on="folder_path", how="left")


df_list_files = df_folders.merge(df_xlsx, on="folder_path", how="left")
display(df_list_files[~df_list_files["folder_name"].str.startswith("00")])

,folder_name,folder_path,file_name,file_path,file_modified
2,01. GELEX,2026/Project Phoenix/04. Working files/01. GELEX,NaN,NaN,NaT
3,02. GEE,2026/Project Phoenix/04. Working files/02. GEE,NaN,NaN,NaT
4,03. CADIVI,2026/Project Phoenix/04. Working files/03. CADIVI,KH_CADIVI_5Y_29Apr2026.xlsx,2026/Project Phoenix/04. Working files/03. CADIVI/KH_CADIVI_5Y_29Apr2026.xlsx,2026-05-04 05:51:35+00:00
5,04. CMB,2026/Project Phoenix/04. Working files/04. CMB,NaN,NaN,NaT
6,05. THI,2026/Project Phoenix/04. Working files/05. THI,KH_THIBIDI_5Y_29Apr2026.xlsm,2026/Project Phoenix/04. Working files/05. THI/KH_THIBIDI_5Y_29Apr2026.xlsm,2026-05-05 01:56:42+00:00
7,06. MEE,2026/Project Phoenix/04. Working files/06. MEE,[MEE]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/06. MEE/[MEE]_CFForecast_2026_2030.xlsx,2026-05-04 01:36:02+00:00
8,07. HEM,2026/Project Phoenix/04. Working files/07. HEM,[HEM]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/07. HEM/[HEM]_CFForecast_2026_2030.xlsx,2026-04-29 08:45:56+00:00
9,08. EMIC,2026/Project Phoenix/04. Working files/08. EMIC,[EMIC]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/08. EMIC/[EMIC]_CFForecast_2026_2030.xlsx,2026-05-05 04:04:12+00:00
10,09. CFT,2026/Project Phoenix/04. Working files/09. CFT,KH_CFT_5Y_29Apr2026.xlsx,2026/Project Phoenix/04. Working files/09. CFT/KH_CFT_5Y_29Apr2026.xlsx,2026-05-04 02:52:32+00:00
11,10. Phatdien,2026/Project Phoenix/04. Working files/10. Phatdien,Phatdien_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/10. Phatdien/Phatdien_CFForecast_2026_2030.xlsx,2026-05-05 01:19:35+00:00


In [8]:
list_path_excel = df_list_files[df_list_files["file_path"].notnull()][
    "file_path"
].tolist()

In [9]:
df_report, df_report_status = await client.read_excel_many(
    list_path_excel,
    sheet="Report",
    on_error="skip",
    on_missing_sheet="skip",
    skiprows=1,
    return_status=True,
    usecols="A:M",
)
# display(df_report.head())
# display(df_report_status)

print(df_report.columns)

Index(['Code', 'Phân cấp đơn vị', 'Mã đơn vị', 'Tên đơn vị', 'Khoản mục',
       'Chỉ tiêu', 'Năm', 'Quý', 'Số tiền tổng',
       'Nguồn thông tin\n(Các ghi chú khác về dữ liệu, nguồn thông tin,...nếu cần để sau có thể follow up',
       'Phân loại ổn định và không ổn định',
       'Phân loại Bên trong và bên ngoài', 'Đối tượng giao dịch kinh tế',
       '_source'],
      dtype='str')


In [10]:

column_mapping = {
    "Code": "code",
    "Phân cấp đơn vị": "phan_cap_don_vi",
    "Mã đơn vị": "ma_don_vi",
    "Tên đơn vị": "ten_don_vi",
    "Khoản mục": "khoan_muc",
    "Chỉ tiêu": "chi_tieu",
    "Năm": "nam",
    "Quý": "quy",
    "Số tiền tổng": "so_tien_tong",
    "Nguồn thông tin\n(Các ghi chú khác về dữ liệu, nguồn thông tin,...nếu cần để sau có thể follow up": "nguon_thong_tin",
    "Phân loại ổn định và không ổn định": "phan_loai_on_dinh_khong_on_dinh",
    "Phân loại Bên trong và bên ngoài": "phan_loai_ben_trong_ben_ngoai",
    "Đối tượng giao dịch kinh tế": "doi_tuong_giao_dich_kinh_te",
    "_source": "source",
}
df_report = df_report.rename(columns=column_mapping)
df_report = df_report[list(column_mapping.values())]
# display(df_report.head())


In [11]:

df_report.columns

Index(['code', 'phan_cap_don_vi', 'ma_don_vi', 'ten_don_vi', 'khoan_muc',
       'chi_tieu', 'nam', 'quy', 'so_tien_tong', 'nguon_thong_tin',
       'phan_loai_on_dinh_khong_on_dinh', 'phan_loai_ben_trong_ben_ngoai',
       'doi_tuong_giao_dich_kinh_te', 'source'],
      dtype='str')

In [12]:
df_key_drivers, df_key_drivers_status = await client.read_excel_many(
    list_path_excel,
    sheet="Key drivers",
    on_error="skip",
    on_missing_sheet="skip",
    skiprows=1,
    return_status=True,
    usecols="A:J",
)
df_key_drivers.columns
# display(df_key_drivers)
# display(df_key_drivers_status)

Index(['Group', 'Code', 'Phân cấp đơn vị', 'Mã đơn vị', 'Tên đơn vị',
       'Chỉ tiêu', 'Cấu phần value chain', 'Key drivers',
       'Phân loại Ổn định & Không ổn định',
       'Rationale ổn định or không ổn định', '_source',
       'Phân loại ổn định và không ổn định',
       'Phân loại Bên trong và bên ngoài'],
      dtype='str')

In [13]:
# Merge the two near-duplicate columns before renaming
df_key_drivers["Phân loại Ổn định & Không ổn định"] = (
    df_key_drivers["Phân loại Ổn định & Không ổn định"]
    .combine_first(df_key_drivers["Phân loại ổn định và không ổn định"])
)
df_key_drivers = df_key_drivers.drop(columns=["Phân loại ổn định và không ổn định"])

column_mapping = {
    "Group": "group",
    "Code": "code",
    "Phân cấp đơn vị": "phan_cap_don_vi",
    "Mã đơn vị": "ma_don_vi",
    "Tên đơn vị": "ten_don_vi",
    "Chỉ tiêu": "chi_tieu",
    "Cấu phần value chain": "cau_phan_value_chain",
    "Key drivers": "key_drivers",
    "Phân loại Ổn định & Không ổn định": "phan_loai_on_dinh_khong_on_dinh",
    "Rationale ổn định or không ổn định": "rationale_on_dinh_khong_on_dinh",
    "Phân loại Bên trong và bên ngoài": "phan_loai_ben_trong_ben_ngoai",
    "_source": "source",
}
df_key_drivers = df_key_drivers.rename(columns=column_mapping)
df_key_drivers = df_key_drivers[list(column_mapping.values())]
# display(df_key_drivers.head())


In [14]:
df_key_drivers.columns

Index(['group', 'code', 'phan_cap_don_vi', 'ma_don_vi', 'ten_don_vi',
       'chi_tieu', 'cau_phan_value_chain', 'key_drivers',
       'phan_loai_on_dinh_khong_on_dinh', 'rationale_on_dinh_khong_on_dinh',
       'phan_loai_ben_trong_ben_ngoai', 'source'],
      dtype='str')

In [15]:
df_ratio_commit, df_ratio_commit_status = await client.read_excel_many(
    list_path_excel,
    sheet="Ratio_commit",
    on_error="skip",
    on_missing_sheet="skip",
    skiprows=1,
    return_status=True,
    usecols="A:J",
)
# display(df_ratio_commit.head())
# display(df_ratio_commit_status)
df_ratio_commit.columns

Index(['Phân cấp đơn vị', 'Mã đơn vị', 'Tên đơn vị', 'Chỉ tiêu cam kết',
       'Công thức', 'Giá trị cam kết', 'Năm', 'Quý', 'Giá trị thực hiện',
       'Status (ok/ break)', '_source', 'Nguồn số liệu', 'Định kỳ'],
      dtype='str')

In [16]:
column_mapping = {
    "Phân cấp đơn vị": "phan_cap_don_vi",
    "Mã đơn vị": "ma_don_vi",
    "Tên đơn vị": "ten_don_vi",
    "Chỉ tiêu cam kết": "chi_tieu_cam_ket",
    "Công thức": "cong_thuc",
    "Giá trị cam kết": "gia_tri_cam_ket",
    "Năm": "nam",
    "Quý": "quy",
    "Giá trị thực hiện": "gia_tri_thuc_hien",
    "Status (ok/ break)": "status",
    "_source": "source",
    "Nguồn số liệu": "nguon_so_lieu",
    "Định kỳ": "dinh_ky",
}
df_ratio_commit = df_ratio_commit.rename(columns=column_mapping)
df_ratio_commit = df_ratio_commit[list(column_mapping.values())]
# display(df_ratio_commit.head())
df_ratio_commit.columns

Index(['phan_cap_don_vi', 'ma_don_vi', 'ten_don_vi', 'chi_tieu_cam_ket',
       'cong_thuc', 'gia_tri_cam_ket', 'nam', 'quy', 'gia_tri_thuc_hien',
       'status', 'source', 'nguon_so_lieu', 'dinh_ky'],
      dtype='str')

In [17]:
df_adl, df_adl_input_status = await client.read_excel_many(
    list_path_excel,
    sheet="ADL input",
    on_error="skip",
    on_missing_sheet="skip",
    skiprows=1,
    return_status=True,
    usecols="A:I",
)
# display(df_adl.head())
# display(df_adl_input_status)

df_adl.columns

Index(['Phân cấp đơn vị', 'Mã đơn vị', 'Tên đơn vị', 'Năm',
       'Giai đoạn ngành\n(Industry Lifecycle) ▼',
       'Vị thế cạnh tranh\n(Competition) ▼', 'Mức độ\ntin cậy ▼',
       'Cơ sở đánh giá / Rationale\n(ngành, DN, market...)',
       'Thị phần\nước tính (%)', '_source'],
      dtype='str')

In [18]:
column_mapping = {
    "Phân cấp đơn vị": "phan_cap_don_vi",
    "Mã đơn vị": "ma_don_vi",
    "Tên đơn vị": "ten_don_vi",
    "Năm": "nam",
    "Giai đoạn ngành\n(Industry Lifecycle) ▼": "giai_doan_nganh",
    "Vị thế cạnh tranh\n(Competition) ▼": "vi_the_canh_tranh",
    "Mức độ\ntin cậy ▼": "muc_do_tin_cay",
    "Cơ sở đánh giá / Rationale\n(ngành, DN, market...)": "co_so_danh_gia",
    "Thị phần\nước tính (%)": "thi_phan_uoc_tinh",
    "_source": "source",
}
df_adl = df_adl.rename(columns=column_mapping)
df_adl = df_adl[list(column_mapping.values())]
# display(df_adl.head())
df_adl.columns

Index(['phan_cap_don_vi', 'ma_don_vi', 'ten_don_vi', 'nam', 'giai_doan_nganh',
       'vi_the_canh_tranh', 'muc_do_tin_cay', 'co_so_danh_gia',
       'thi_phan_uoc_tinh', 'source'],
      dtype='str')

In [19]:
df_summary = df_list_files.copy()

for status_df, col in [
    (df_report_status, "report"),
    (df_key_drivers_status, "key_drivers"),
    (df_ratio_commit_status, "ratio_commit"),
    (df_adl_input_status, "adl_input"),
]:
    df_summary = df_summary.merge(
        status_df[["path", "status"]].rename(
            columns={
                "path": "file_path",
                "status": f"{col}_status",
            }
        ),
        on="file_path",
        how="left",
    )

display(df_summary[~df_summary["folder_name"].str.startswith("00")])

,folder_name,folder_path,file_name,file_path,file_modified,report_status,key_drivers_status,ratio_commit_status,adl_input_status
2,01. GELEX,2026/Project Phoenix/04. Working files/01. GELEX,NaN,NaN,NaT,NaN,NaN,NaN,NaN
3,02. GEE,2026/Project Phoenix/04. Working files/02. GEE,NaN,NaN,NaT,NaN,NaN,NaN,NaN
4,03. CADIVI,2026/Project Phoenix/04. Working files/03. CADIVI,KH_CADIVI_5Y_29Apr2026.xlsx,2026/Project Phoenix/04. Working files/03. CADIVI/KH_CADIVI_5Y_29Apr2026.xlsx,2026-05-04 05:51:35+00:00,success,success,success,success
5,04. CMB,2026/Project Phoenix/04. Working files/04. CMB,NaN,NaN,NaT,NaN,NaN,NaN,NaN
6,05. THI,2026/Project Phoenix/04. Working files/05. THI,KH_THIBIDI_5Y_29Apr2026.xlsm,2026/Project Phoenix/04. Working files/05. THI/KH_THIBIDI_5Y_29Apr2026.xlsm,2026-05-05 01:56:42+00:00,success,success,success,success
7,06. MEE,2026/Project Phoenix/04. Working files/06. MEE,[MEE]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/06. MEE/[MEE]_CFForecast_2026_2030.xlsx,2026-05-04 01:36:02+00:00,success,success,success,missing_sheet
8,07. HEM,2026/Project Phoenix/04. Working files/07. HEM,[HEM]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/07. HEM/[HEM]_CFForecast_2026_2030.xlsx,2026-04-29 08:45:56+00:00,success,success,success,missing_sheet
9,08. EMIC,2026/Project Phoenix/04. Working files/08. EMIC,[EMIC]_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/08. EMIC/[EMIC]_CFForecast_2026_2030.xlsx,2026-05-05 04:04:12+00:00,success,success,missing_sheet,missing_sheet
10,09. CFT,2026/Project Phoenix/04. Working files/09. CFT,KH_CFT_5Y_29Apr2026.xlsx,2026/Project Phoenix/04. Working files/09. CFT/KH_CFT_5Y_29Apr2026.xlsx,2026-05-04 02:52:32+00:00,success,success,success,success
11,10. Phatdien,2026/Project Phoenix/04. Working files/10. Phatdien,Phatdien_CFForecast_2026_2030.xlsx,2026/Project Phoenix/04. Working files/10. Phatdien/Phatdien_CFForecast_2026_2030.xlsx,2026-05-05 01:19:35+00:00,success,success,success,missing_sheet
